# Welcome to the notebook on tuning an adapter for Gemma!

We will tune adapters that generate memes in a specific format.

What is required for this notebook to run:


*   Your Hugging Face token to get access to Inference API and be able to push the adapters.
*   Mounting your drive in this notebook.
*   It's best to run this notebook with GPU (e.g., Google Colab's T4 GPUs).



In [ ]:
hf_token = 'hf_YOUR_TOKEN'

In [ ]:
# login using your token
from huggingface_hub import notebook_login

In [ ]:
notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Load the necessary libraries and the model - Gemma2B


In [ ]:
!pip3 install peft
!pip3 install -q -U bitsandbytes==0.42.0
!pip3 install -q -U peft==0.8.2
!pip3 install -q -U trl==0.7.10
!pip3 install -q -U accelerate==0.27.1
!pip3 install -q -U datasets==2.17.0
!pip3 install -q -U transformers==4.38.0

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import os

In [ ]:
model_id = "google/gemma-2b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0}, token=hf_token)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Load data

In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset

Here you can load the dataframe with your memes. The dataframe consists of captions and topics.

How I obtain these dataframes:

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/meme_caption_generation/df_angry.csv")

In [ ]:
text_column = "topic"
label_column = "caption"

In [ ]:
# Create Transformers dataset
data = {"topic": [word for word in df['topic']], "caption":[descr for descr in df['caption']]}
dataset = Dataset.from_dict(data)
dataset_split = dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)
dataset_split['train'][0]
data_dict = Dataset.from_dict(data)

In [ ]:
def generate_prompt(data_point):
    """Gen. input text based on a prompt, task instruction, (context info.), and answer

    :param data_point: dict: Data point
    :return: dict: tokenzed prompt
    """
    prefix_text = '''Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n
                  You are given a topic. Your task is to generate a meme caption based on the topic. Only output the meme caption and nothing more.'''

    text = f"""<start_of_turn>user {prefix_text} Topic: {data_point["topic"]}<end_of_turn>\\n<start_of_turn>model Caption: {data_point['caption']} <end_of_turn>"""
    return text

In [ ]:
# add the "prompt" column in the dataset
text_column = [generate_prompt(data_point) for data_point in data_dict]

In [ ]:
text_column[0]

"<start_of_turn>user Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n\n                  You are given a topic. Your task is to generate a meme caption based on the topic. Only output the meme caption and nothing more. Topic: a person asking a rhetorical question about someone's interests<end_of_turn>\\n<start_of_turn>model Caption: NICK GIANELLA Y U ONLY LIKE CARS N GUNS?! <end_of_turn>"

In [ ]:
data_dict = data_dict.add_column("prompt", text_column)

In [ ]:
dataset = data_dict.shuffle(seed=1234)  # Shuffle dataset here
dataset = dataset.map(lambda samples: tokenizer(samples["prompt"]), batched=True)

Map:   0%|          | 0/625 [00:00<?, ? examples/s]

In [ ]:
dataset = dataset.train_test_split(test_size=0.2)
train_data = dataset["train"]
test_data = dataset["test"]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['topic', 'caption', 'prompt', 'input_ids', 'attention_mask'],
        num_rows: 500
    })
    test: Dataset({
        features: ['topic', 'caption', 'prompt', 'input_ids', 'attention_mask'],
        num_rows: 125
    })
})

In [ ]:
from peft import LoraConfig, get_peft_model

In [ ]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=32,
    target_modules=['o_proj', 'q_proj', 'up_proj', 'v_proj', 'k_proj', 'down_proj', 'gate_proj'],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

In [ ]:
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable} | total: {total} | Percentage: {trainable/total*100:.4f}%")


Trainable: 78446592 | total: 2584619008 | Percentage: 3.0351%


In [ ]:
import transformers
from trl import SFTTrainer

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
torch.cuda.empty_cache()
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=test_data,
    dataset_text_field="prompt",
    peft_config=lora_config,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=0.03,
        max_steps=100,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir="/content/drive/MyDrive/meme_caption_generation/outputs_gemma2b_angry",
        optim="paged_adamw_8bit",
        save_strategy="epoch",
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:223: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:290: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(


In [ ]:
# Start the training process
trainer.train()

Step,Training Loss
1,5.050800
2,5.019500
3,4.436500
4,3.999000
5,3.827600
6,3.523400
7,2.963900
8,2.565400
9,2.022700
10,1.905300


Checkpoint destination directory /content/drive/MyDrive/BMW/outputs_gemma2b_angry/checkpoint-100 already exists and is non-empty. Saving will proceed but saved results may be invalid.


TrainOutput(global_step=100, training_loss=1.431666259765625, metrics={'train_runtime': 273.8296, 'train_samples_per_second': 1.461, 'train_steps_per_second': 0.365, 'total_flos': 400676691787776.0, 'train_loss': 1.431666259765625, 'epoch': 0.8})

In [ ]:
new_model = "gemma-2bit-happy_memes" #Name of the model you will be pushing to huggingface model hub
# Save the fine-tuned model
trainer.model.save_pretrained(new_model)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# Merge the model with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    'google/gemma-2b',
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from peft import LoraConfig, PeftModel

In [ ]:
merged_model= PeftModel.from_pretrained(base_model, new_model)
merged_model= merged_model.merge_and_unload()

In [ ]:
# # Save the merged model (optionally)
# merged_model.save_pretrained("merged_model_happy",safe_serialization=True)
# tokenizer.save_pretrained("merged_model_happy")
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

In [ ]:
# # Push the model and tokenizer to the Hugging Face Model Hub (also optionally)
# merged_model.push_to_hub(new_model, use_temp_dir=False)
# tokenizer.push_to_hub(new_model, use_temp_dir=False)

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/NursNurs/gemma-2bit-y_u_no_memes/commit/f2ccd8e24e6475113e2ffbad394084d0d48fb83f', commit_message='Upload tokenizer', commit_description='', oid='f2ccd8e24e6475113e2ffbad394084d0d48fb83f', pr_url=None, pr_revision=None, pr_num=None)

In [ ]:
def get_completion(query: str, model, tokenizer) -> str:
  device = "cuda:0"
  prompt_template = """
  Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n
  You are given a topic. Your task is to generate a meme caption based on the topic. Only output the meme caption and nothing more.
  Topic: {query}
  <end_of_turn>\\n<start_of_turn>model Caption:

  """
  prompt = prompt_template.format(query=query)
  encodeds = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
  model_inputs = encodeds.to(device)
  generated_ids = model.generate(**model_inputs, max_new_tokens=20, do_sample=True, pad_token_id=tokenizer.eos_token_id)
  # decoded = tokenizer.batch_decode(generated_ids)
  decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
  return (decoded)



In [ ]:
result = get_completion(query="frienda eating lunch", model=merged_model, tokenizer=tokenizer)
print(result)


  Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n
  You are given a topic. Your task is to generate a meme caption based on the topic. Only output the meme caption and nothing more.
  Topic: frienda eating lunch
  Caption:

  Girlie, i wanna come over n eat lunch.  
  You: y u no come


In [ ]:
result.split('Caption:')[1]

'\n\n  Girlie, i wanna come over n eat lunch.  \n  You: y u no come'

In [ ]:
trainer.push_to_hub("NursNurs/gemma2-angry-memes")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Upload 6 LFS files:   0%|          | 0/6 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

events.out.tfevents.1726332186.9adda4255a64.81023.1:   0%|          | 0.00/26.2k [00:00<?, ?B/s]

events.out.tfevents.1726331696.9adda4255a64.81023.0:   0%|          | 0.00/26.2k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/314M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/NursNurs/outputs_gemma2b_angry/commit/068d260e5888dbb3cfb66f81ee6d7913d3f86bb8', commit_message='NursNurs/gemma2-angry-memes', commit_description='', oid='068d260e5888dbb3cfb66f81ee6d7913d3f86bb8', pr_url=None, pr_revision=None, pr_num=None)

In [ ]:
data_example = load_dataset("Abirate/english_quotes")

In [ ]:
data_example = data_example.map(lambda samples: tokenizer(samples["quote"]), batched=True)

In [ ]:
data_example

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
        num_rows: 2508
    })
})

In [ ]:
dataset_split = data_dict.train_test_split(test_size=0.2, shuffle=True, seed=42)

In [ ]:
dataset_split

DatasetDict({
    train: Dataset({
        features: ['topic', 'caption', 'input_ids', 'attention_mask'],
        num_rows: 245
    })
    test: Dataset({
        features: ['topic', 'caption', 'input_ids', 'attention_mask'],
        num_rows: 62
    })
})

In [ ]:
def formatting_func(example):
    text = f"Topic: {example['topic'][0]}\nCaption: {example['caption'][0]}<eos>"
    return [text]

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_split["train"],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=10,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="outputs_gemma2b_yuso",
        optim="paged_adamw_8bit"
    ),
    peft_config=lora_config,
    formatting_func=formatting_func,
)
trainer.train()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:223: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/245 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:290: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
1,1.471900
2,1.471900
3,1.429900
4,1.330400
5,1.236800
6,1.150200
7,1.074900
8,1.014000
9,0.967800
10,0.938100


TrainOutput(global_step=10, training_loss=1.208595633506775, metrics={'train_runtime': 4.1443, 'train_samples_per_second': 9.652, 'train_steps_per_second': 2.413, 'total_flos': 2629031116800.0, 'train_loss': 1.208595633506775, 'epoch': 10.0})

In [ ]:
text = "Topic: a man complimenting someone"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Topic: a man complimenting someone on their appearance

Context:

A man is complimenting a woman on her appearance.

Question
